# 03. 양자화 — 같은 모델을 4분의 1 크기로

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/gunyang-ai-api/blob/main/labs/04_quantization.ipynb)

> 🧭 **스토리** — 팀장의 다음 질문. "사내 서버 GPU가 작은데, 더 큰 모델은 못 올리나?" 같은 모델을 **4비트**로 줄여 메모리·속도·답을 비교하고, T4(15GB)에 **4B 모델**까지 올려 본다.
>
> 📖 **이론 연결** — 이론1 p62~67 (오픈 모델 크기·정밀도·양자화, 로컬 서빙 비용) · ⏱ **10분** · 🖥 **GPU 필수**
>
> **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택하세요. (GPU가 없으면 작은 설정으로 느리게 돌아갑니다.)
> 💡 **앞 노트북을 열어 둔 상태라면** 그 노트북에서 `런타임 → 런타임 연결 해제 및 삭제`를 먼저 하세요. 무료 코랩은 GPU 세션을 하나만 허용합니다.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 0. 준비 ⚙️
`bitsandbytes`는 허깅페이스 transformers에 붙는 양자화 라이브러리입니다. **CUDA GPU 전용**이라 CPU 런타임에서는 4비트·8비트 셀을 건너뜁니다(안내 문구가 나옵니다).

In [ ]:
%pip -q install -U "transformers>=4.54" accelerate bitsandbytes

In [ ]:
# ⚙️ GPU 확인 — T4가 잡혔는지, 메모리가 얼마인지
import torch, subprocess
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {name} · 메모리 {total_gb:.1f} GB")
else:
    print("GPU 없음 — CPU로 진행합니다(느립니다). 런타임 유형에서 T4를 선택하면 빨라집니다.")
DEVICE = "cuda" if HAS_GPU else "cpu"
DTYPE = torch.float16 if HAS_GPU else torch.float32   # T4는 bf16을 지원하지 않아 fp16을 씁니다

## 1. 기준선 — fp16 원본 🔵
노트북 03와 같은 EXAONE 4.0 1.2B를 fp16으로 올리고 **같은 질문 3개**에 답하게 합니다. 기록할 것은 세 가지입니다.
- **메모리**: 가중치 크기(`get_memory_footprint`)와 생성 중 최대 GPU 사용량(`max_memory_allocated`)
- **속도**: 초당 생성 토큰(토큰/초)
- **답**: greedy(`do_sample=False`)로 생성해 설정끼리 글자 단위로 비교

In [ ]:
import os, gc, time
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SMOKE = os.environ.get("NB_SMOKE") == "1"          # 강사 검증용 스위치 — 수강생은 신경 쓰지 않아도 됩니다
if SMOKE:
    MODEL_NAME, MAX_NEW = "Qwen/Qwen2.5-0.5B-Instruct", 30
elif HAS_GPU:
    MODEL_NAME, MAX_NEW = "LGAI-EXAONE/EXAONE-4.0-1.2B", 60
else:                                             # CPU: 같은 모델을 fp32로, 생성은 짧게
    MODEL_NAME, MAX_NEW = "LGAI-EXAONE/EXAONE-4.0-1.2B", 30

QUESTIONS = ["국내 출장 숙박비는 어떻게 정산하나요? 두 문장으로 답하세요.",
             "양자화가 무엇인지 중학생에게 한 문장으로 설명하세요.",
             "한빛테크 신입 사원이 첫 출장 전에 확인할 것 세 가지를 짧게 쓰세요."]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def load(quant_config=None):
    """같은 모델을 정밀도 설정만 바꿔 로드합니다. quant_config=None 이면 원본(fp16/fp32)."""
    if HAS_GPU:
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, device_map="auto" if HAS_GPU else None,
                                             quantization_config=quant_config)
    print(f"로드 {time.time()-t0:.0f}초 · 가중치 {m.get_memory_footprint()/1e9:.2f} GB")
    return m

def bench(model, tag, max_new=None):
    """3개 질문을 greedy로 생성하고 (요약 1행, 답 리스트)를 돌려줍니다."""
    answers, n_tok, sec = [], 0, 0.0
    for q in QUESTIONS:
        inputs = tokenizer.apply_chat_template([{"role": "user", "content": q}], add_generation_prompt=True,
                                               return_tensors="pt", return_dict=True,
                                               enable_thinking=False).to(model.device)
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new or MAX_NEW, do_sample=False)
        sec += time.time() - t0
        new = out[0, inputs["input_ids"].shape[1]:]
        n_tok += len(new)
        answers.append(tokenizer.decode(new, skip_special_tokens=True).strip())
    row = {"설정": tag,
           "가중치 GB": round(model.get_memory_footprint() / 1e9, 2),
           "최대 GPU 메모리 GB": round(torch.cuda.max_memory_allocated() / 1e9, 2) if HAS_GPU else None,
           "토큰/초": round(n_tok / sec, 1),
           "평균 답 길이(토큰)": round(n_tok / len(QUESTIONS), 1)}
    return row, answers

model = load()                                     # 원본 정밀도
row, base_answers = bench(model, "fp16 원본" if HAS_GPU else "fp32 원본(CPU)")
results = [row]
print(row)
for q, a in zip(QUESTIONS, base_answers):
    print(f"\nQ. {q}\nA. {a}")

💡 **해석** — 이 행이 **기준선**입니다. T4에서 EXAONE 1.2B fp16의 가중치는 약 2.56GB(1.28B 파라미터 × 2바이트)이고, 생성 중 최대 메모리는 그보다 조금 큽니다(입력·KV 캐시·중간 계산). 토큰/초는 T4에서 보통 수십 토큰 수준입니다. 이 숫자를 기억하고 다음 셀과 비교하세요.

## 2. GPU 메모리 비우기 🔵
무료 T4는 메모리가 약 15GB뿐입니다. 다음 모델을 올리기 전에 **참조 삭제 → 가비지 컬렉션 → CUDA 캐시 비우기**를 순서대로 합니다. 실무에서도 모델을 갈아 끼울 때 똑같이 합니다.

In [ ]:
del model
gc.collect()
if HAS_GPU:
    torch.cuda.empty_cache()
    print(f"비운 뒤 GPU 메모리: {torch.cuda.memory_allocated()/1e9:.2f} GB")
else:
    print("CPU 메모리 해제 완료")

## 3. 4비트(NF4)로 다시 로드 🔵
[허깅페이스 bitsandbytes 문서](https://huggingface.co/docs/transformers/main/en/quantization/bitsandbytes)의 설정 그대로입니다.
- `load_in_4bit=True`: 가중치를 4비트로 저장
- `bnb_4bit_quant_type="nf4"`: 정규분포 모양의 가중치에 맞춘 4비트 형식(NormalFloat4)
- `bnb_4bit_compute_dtype=torch.float16`: 계산할 때는 fp16으로 풀어서 곱합니다. **T4는 bf16을 지원하지 않으므로 fp16**입니다.

In [ ]:
if HAS_GPU:
    bnb4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.float16)
    model = load(bnb4)
    row, q4_answers = bench(model, "4bit NF4")
    row["답 동일(원본 대비)"] = f"{sum(a == b for a, b in zip(q4_answers, base_answers))}/{len(QUESTIONS)}"
    results.append(row)
    for q, a in zip(QUESTIONS, q4_answers):
        print(f"\nQ. {q}\nA. {a}")
    del model; gc.collect(); torch.cuda.empty_cache()      # 다음 실험을 위해 바로 비웁니다
else:
    print("(건너뜀) bitsandbytes 4비트는 CUDA GPU에서만 동작합니다. 런타임 유형을 T4로 바꾸고 다시 실행하세요.")

## 4. 비교 표 🔵

In [ ]:
results[0]["답 동일(원본 대비)"] = "기준"
df = pd.DataFrame(results).set_index("설정")
display(df)
if len(df) > 1:
    r = df["가중치 GB"].iloc[0] / df["가중치 GB"].iloc[1]
    print(f"가중치 크기: 원본의 1/{r:.1f}")

💡 **해석** — 표에서 세 가지를 확인하세요.
- **메모리**: 4비트 가중치는 원본의 **1/2~1/3 정도**로 줄어듭니다. 이론상 1/4(16비트 → 4비트)인데 덜 줄어드는 이유는 **임베딩·출력층(`lm_head`)은 양자화하지 않기** 때문입니다. EXAONE 1.2B는 어휘 10만 × 2048차원 임베딩이 약 2억 파라미터로 전체의 6분의 1이라, 작은 모델일수록 이 몫이 큽니다. 모델이 클수록 4배에 가까워집니다.
- **속도**: bitsandbytes 4비트는 **빨라지지 않습니다**. 계산할 때마다 4비트를 fp16으로 풀기 때문에 같거나 조금 느린 경우가 많습니다. bitsandbytes의 목적은 속도가 아니라 **메모리 절약**입니다.
- **답**: greedy인데도 답이 원본과 글자 단위로 달라질 수 있습니다. 가중치가 조금 바뀌면 어느 시점에서 1등 토큰이 바뀌고, 그 뒤 문장 전체가 갈라집니다. 뜻이 같은지 직접 읽어 보세요.

## 5. 심화 🟠 (시간 남을 때) — 8비트, 그리고 긴 생성
1. 아래 셀은 `load_in_8bit=True`(LLM.int8)로 로드해 표에 한 줄을 더합니다. 메모리가 4비트와 원본 사이에 오는지 보세요.
2. `MAX_NEW_TRY`를 **200**으로 바꿔 다시 실행하면 긴 생성에서 속도 차이가 더 분명해집니다.

In [ ]:
MAX_NEW_TRY = MAX_NEW                                   # ← 200 으로 바꿔 보세요
TRY_CFG = BitsAndBytesConfig(load_in_8bit=True)        # ← 4비트 설정(bnb4)으로 바꿔 비교해도 됩니다
if HAS_GPU:
    model = load(TRY_CFG)
    row, ans = bench(model, f"8bit · {MAX_NEW_TRY}토큰", max_new=MAX_NEW_TRY)
    row["답 동일(원본 대비)"] = f"{sum(a == b for a, b in zip(ans, base_answers))}/{len(QUESTIONS)}"
    results.append(row)
    display(pd.DataFrame(results).set_index("설정"))
    del model; gc.collect(); torch.cuda.empty_cache()
else:
    print("(건너뜀) bitsandbytes 8비트도 CUDA GPU 전용입니다.")

## 6. T4에 4B 모델 올리기 🔵 — 사전 양자화 모델
4비트로 줄이면 **같은 GPU에 더 큰 모델**이 들어갑니다. `unsloth/Qwen3-4B-unsloth-bnb-4bit`는 Qwen3-4B(Apache 2.0)를 **미리 4비트로 저장해 둔** 모델입니다. 로그인·동의 없이 받을 수 있고 다운로드는 약 3.6GB입니다(1~2분).
저장된 설정의 계산 dtype이 bf16이라 T4용으로 **fp16으로 바꿔서** 로드합니다. 시간이 부족하면 이 절은 건너뛰고 7절을 읽으세요.

In [ ]:
from transformers import AutoConfig
Q4B = "unsloth/Qwen3-4B-unsloth-bnb-4bit"
if HAS_GPU:
    cfg = AutoConfig.from_pretrained(Q4B)
    cfg.quantization_config["bnb_4bit_compute_dtype"] = "float16"     # 저장값 bfloat16 → T4는 fp16
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    tok4 = AutoTokenizer.from_pretrained(Q4B)
    model4 = AutoModelForCausalLM.from_pretrained(Q4B, config=cfg, dtype=torch.float16, device_map="auto")
    print(f"로드 {time.time()-t0:.0f}초 · 파라미터 {sum(p.numel() for p in model4.parameters())/1e9:.2f}B(4비트 묶음 기준)"
          f" · 가중치 {model4.get_memory_footprint()/1e9:.2f} GB")

    msgs = [{"role": "user", "content": QUESTIONS[0]}]
    inputs = tok4.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True,
                                      enable_thinking=False).to(model4.device)   # Qwen3도 사고 모드를 끕니다
    t0 = time.time()
    with torch.no_grad():
        out = model4.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=False)
    new = out[0, inputs["input_ids"].shape[1]:]
    print(f"생성 {len(new)/(time.time()-t0):.1f} 토큰/초 · 최대 GPU 메모리 {torch.cuda.max_memory_allocated()/1e9:.2f} GB"
          f" / 전체 {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print("\nQ.", QUESTIONS[0], "\nA.", tok4.decode(new, skip_special_tokens=True).strip())
else:
    print("(건너뜀) 사전 양자화 bnb-4bit 모델도 CUDA GPU가 필요합니다.")

💡 **해석** — Qwen3-4B를 fp16으로 올리면 가중치만 약 8GB입니다. 4비트로는 **3GB 안팎**이라 T4에 넉넉히 들어가고, 1.2B 원본(2.56GB)과 비슷한 메모리로 **파라미터가 3배 넘는 모델**을 쓰는 셈입니다. 답의 품질이 1.2B와 어떻게 다른지 1절의 첫 번째 답과 비교해 보세요.

## 7. GGUF — 노트북 05로 가는 다리 📖
bitsandbytes는 **파이썬·transformers·CUDA 안에서** 쓰는 양자화입니다. 반면 **GGUF**는 llama.cpp 계열 엔진(Ollama, LM Studio 등)이 쓰는 **파일 형식**입니다. 양자화된 가중치와 토크나이저·채팅 템플릿을 파일 하나에 담아서 **CPU·맥·윈도우에서도** 돌릴 수 있습니다.
- 이름의 `Q4_K_M`은 "4비트, K-quant 방식, 중간(M) 품질"이라는 뜻입니다. EXAONE 4.0 1.2B의 `Q4_K_M` GGUF는 **812MB**입니다(fp16 2.56GB의 약 1/3).
- 노트북 05에서 이 파일을 Ollama로 받아 **OpenAI와 똑같은 코드**로 부릅니다.
- AWQ·GPTQ(사전 양자화 형식)와 vLLM·`transformers serve`(서빙 엔진)는 설치가 무거워 장표에서 소개만 합니다.

**공식 문서에 나온 숫자**

| 비교 | 결과 | 출처 |
|---|---|---|
| Qwen3-8B **AWQ 4비트** vs **bf16** (MMLU-Redux) | 86.4 vs 87.5 (1.1점 하락) | Qwen3-8B-AWQ 모델 카드 |
| bitsandbytes **4비트** 메모리 | 원본 대비 약 **4배 절감** | HF bitsandbytes 문서 |
| EXAONE 4.0 1.2B 파일 크기 | fp16 2.56GB → GGUF Q4_K_M 812MB | 각 모델 저장소 |

### 🤔 생각해보기

**4비트 4B 모델과 fp16 1.2B 모델, 어느 쪽이 나은가?** 무엇으로 판단해야 할까요?

<details><summary><b>🤔 생각해보기 — 해설 보기(클릭)</b></summary>

정답은 하나가 아닙니다. **같은 메모리 예산**에서 무엇을 사느냐의 문제입니다.
- **품질**: 일반적으로 "큰 모델을 양자화"하는 쪽이 "작은 모델 원본"보다 성능이 좋습니다(Qwen3-8B는 4비트로 줄여도 MMLU-Redux가 1.1점만 떨어짐). 하지만 **우리 업무 질문**으로 직접 재야 합니다. 규정 질문 20개를 만들어 두 모델의 정답률을 비교하세요.
- **속도**: 4B는 계산량이 3배 이상이고 bitsandbytes는 풀기 비용까지 있어 느립니다. 응답 시간 요구(예: 2초 안)가 있으면 1.2B가 유리할 수 있습니다.
- **라이선스**: EXAONE 1.2B는 비상업(NC) 라이선스라 회사 서비스에는 쓸 수 없습니다. Qwen3는 Apache 2.0입니다. **실무에서는 이것이 먼저 걸러지는 조건**입니다.
- **한국어**: 한국어 토큰 효율(노트북 02)과 한국어 답의 자연스러움도 판단 기준입니다.

</details>

---
### ✅ 이 노트북에서 가져갈 것
- 양자화는 가중치의 비트 수를 줄여 메모리를 아낀다. 4비트 NF4 + fp16 계산이 T4의 기본 조합이다.
- 작은 모델은 임베딩·출력층 몫이 커서 4배보다 덜 줄어든다. bitsandbytes는 속도가 아니라 메모리를 위한 도구다.
- greedy라도 양자화하면 답이 글자 단위로 달라질 수 있다. 품질은 업무 질문으로 직접 잰다.
- 4비트로 줄이면 T4에 4B 모델이 들어간다. GGUF는 같은 아이디어를 파일 형식으로 담아 CPU에서도 돌린다.
- 다음 노트북으로 가기 전 `런타임 → 런타임 연결 해제 및 삭제`로 GPU를 비우세요.

> 다음 노트북: **05_openai_compatible_serving — 같은 코드로 로컬 모델과 OpenAI를 번갈아 부르기**